# IA³ fine-tuning — Qwen2.5-Coder-3B on Spider

Single IA³ experiment using the same Qwen/Spider pipeline as the LoRA experiments.

**Configuration**
- Qwen2.5-Coder-3B-Instruct
- IA³ targets: `k_proj`, `v_proj`, `down_proj`
- feed-forward IA³ target: `down_proj`
- 3 epochs
- learning rate: `1e-3`
- batch size: 1
- gradient accumulation: 16
- max length: 2048


In [ ]:
!nvidia-smi


Mon Aug 10 10:51:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             51W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
from google.colab import userdata
from urllib.parse import quote
import shutil
import subprocess

username = userdata.get("GITLAB_USERNAME")
token = userdata.get("GITLAB_TOKEN")

repo_dir = "/content/msc_project"
shutil.rmtree(repo_dir, ignore_errors=True)

url = (
    f"https://{quote(username, safe='')}:{quote(token, safe='')}"
    f"@git.cs.bham.ac.uk/txh517/msc_project.git"
)

subprocess.run(["git", "clone", url, repo_dir], check=True)
print("Repository cloned successfully.")


Repository cloned successfully.


In [ ]:
%cd /content/msc_project
%pip install -q transformers datasets peft trl accelerate
%pip uninstall -y torchao


/content/msc_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 54.4 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
from pathlib import Path
import shutil
import zipfile

zip_path = Path("/content/drive/MyDrive/spider_data.zip")
spider_dir = Path("/content/msc_project/data/spider")

shutil.rmtree(spider_dir, ignore_errors=True)
spider_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(spider_dir)

nested = spider_dir / "spider_data"
if nested.exists():
    for item in nested.iterdir():
        dest = spider_dir / item.name
        if dest.exists():
            if dest.is_dir():
                shutil.rmtree(dest)
            else:
                dest.unlink()
        shutil.move(str(item), str(dest))
    nested.rmdir()

config_path = Path("/content/msc_project/config.yaml")
text = config_path.read_text(encoding="utf-8")
text = text.replace("device: cpu", "device: cuda")
config_path.write_text(text, encoding="utf-8")

required = [
    spider_dir / "train_spider.json",
    spider_dir / "dev.json",
    spider_dir / "tables.json",
    spider_dir / "database",
]

for path in required:
    print(path, "->", path.exists())

print("\nIA3 files:")
print("train_ia3.py ->", Path("experiments/train_ia3.py").exists())
print("run_slm_ia3.py ->", Path("experiments/run_slm_ia3.py").exists())


/content/msc_project/data/spider/train_spider.json -> True
/content/msc_project/data/spider/dev.json -> True
/content/msc_project/data/spider/tables.json -> True
/content/msc_project/data/spider/database -> True

IA3 files:
train_ia3.py -> True
run_slm_ia3.py -> True


In [ ]:
%cd /content/msc_project

!python3 -m experiments.train_ia3 \
    --output_dir "/content/drive/MyDrive/msc_text_to_sql/models/qwen_spider_ia3_e3" \
    --epochs 3 \
    --batch_size 1 \
    --gradient_accumulation 16 \
    --learning_rate 1e-3 \
    --max_length 2048 \
    --save_steps 100


/content/msc_project
Training examples: 7000
Base model: Qwen/Qwen2.5-Coder-3B-Instruct
GPU: NVIDIA A100-SXM4-40GB
config.json: 100% 661/661 [00:00<00:00, 3.32MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 20.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 49.9MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 102MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 144MB/s]
Token lengths: median=2, p95=2, max=2
Sequences over max_length=2048: 0/7000 (0.0%)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 77.8MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  27% 1.64G/6.17G [00:03<00:08, 549MB/s, 67.6MB/s  ]
Reconstructing (incomplete total...): 100% 6.17G/6.17G [00:14<00:00, 422MB/s,  269MB/s  ]

Fetchi

In [ ]:
import json

metadata_path = (
    "/content/drive/MyDrive/msc_text_to_sql/models/"
    "qwen_spider_ia3_e3/final_adapter/training_metadata.json"
)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Training loss:", metadata["training"]["train_loss"])
print("Trainable parameters:", f'{metadata["efficiency"]["trainable_parameters"]:,}')
print("Trainable percentage:", f'{metadata["efficiency"]["trainable_percentage"]:.4f}%')
print("Peak allocated VRAM:", f'{metadata["efficiency"]["peak_gpu_memory_allocated_gb"]:.2f} GB')
print("Peak reserved VRAM:", f'{metadata["efficiency"]["peak_gpu_memory_reserved_gb"]:.2f} GB')

throughput = metadata["efficiency"]["training_tokens_per_second"]
if throughput is not None:
    print("Training throughput:", f"{throughput:.2f} tokens/s")


Training loss: 0.14295147693184726
Trainable parameters: 414,720
Trainable percentage: 0.0134%
Peak allocated VRAM: 7.90 GB
Peak reserved VRAM: 8.51 GB
Training throughput: 1434.96 tokens/s


In [ ]:
%cd /content/msc_project

!python3 -m experiments.run_slm_ia3 \
    --adapter "/content/drive/MyDrive/msc_text_to_sql/models/qwen_spider_ia3_e3/final_adapter" \
    --output "/content/drive/MyDrive/msc_text_to_sql/results/slm_ia3_e3.json" \
    --save_every 10


/content/msc_project
Loading base model Qwen/Qwen2.5-Coder-3B-Instruct ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:01<00:00, 219.37it/s]
Evaluating 1034 examples (checkpointing every 10) ...
[10/1034] running EX=0.800 (0.89 ex/s, ~19.3 min remaining)
[20/1034] running EX=0.750 (0.99 ex/s, ~17.1 min remaining)
[30/1034] running EX=0.800 (0.81 ex/s, ~20.6 min remaining)
[40/1034] running EX=0.800 (0.66 ex/s, ~25.2 min remaining)
[50/1034] running EX=0.740 (0.64 ex/s, ~25.6 min remaining)
[60/1034] running EX=0.700 (0.57 ex/s, ~28.5 min remaining)
[70/1034] running EX=0.614 (0.50 ex/s, ~31.9 min remaining)
[80/1034] running EX=0.625 (0.50 ex/s, ~31.8 min remaining)
[90/1034] running EX=0.633 (0.50 ex/s, ~31.5 min remaining)
[100/1034] running EX=0.620 (0.50 ex/s, ~30.9 min remaining)
[110/1034] running EX=0.573 (0.50 ex/s, ~30.8 min remaining)
[120/1034] running EX=0.550 (0.50 ex/s, ~30.6 min remaining)
[130/1034] running EX=0.546

In [ ]:
import json

result_path = (
    "/content/drive/MyDrive/msc_text_to_sql/results/"
    "slm_ia3_e3.json"
)

with open(result_path, "r", encoding="utf-8") as f:
    results = json.load(f)

correct = sum(bool(x["correct"]) for x in results)
total = len(results)

print(f"Examples: {total}")
print(f"Correct: {correct}")
print(f"Execution accuracy: {correct / total:.2%}")


Examples: 1034
Correct: 732
Execution accuracy: 70.79%
